## 1. Training the PAIDF AnomalyGen modules

You must complete the previous setup step [0-setup-cuda128.ipynb](./0-setup-cuda128.ipynb).

**Important**: This is a quick example, so the generated image quality may be suboptimal. To improve quality, increase the number of training iterations.

**Note**: For full training, it is recommended to use the command line instead of the notebook, as notebook cells have limited logging capacity.

### 1.0 Setting Up the Environment

This notebook requires the user to set the environment variable `LOCAL_PROJECT_DIR` to the path of the PAIDF AnomalyGen repo. Remember to replace `FIXME` placeholder below with the correct path.

In [ ]:
# Set `LOCAL_PROJECT_DIR` for PAIDF AnomalyGen.
LOCAL_PROJECT_DIR="FIXME"
# Set the working directory to this path for the shell.
%cd {LOCAL_PROJECT_DIR}
# Use `cd ${LOCAL_PROJECT_DIR}` if you are copy-pasting this into a terminal.

import os

os.environ["LD_LIBRARY_PATH"] = (
    f"{LOCAL_PROJECT_DIR}/anaconda3/envs/cosmos-predict2/lib/python3.12/site-packages/nvidia/cudnn/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)
os.environ.pop("MPLBACKEND", None)

### 1.1 Providing the Training Configuration

We provide the config file for training in `ag_configs/`:
- 2B: `MeiweiPCB_NVDINOV2_2B_512.yaml`
- 14B: `MeiweiPCB_NVDINOV2_14B_512.yaml`

We will use the 2B model configuration for the following example.

#### 1.1.1 Dataset Configuration

Before starting training, please modify the job section to specify your desired output path:

```yaml
    job:
        project: anomaly_gen
        group: MeiweiPCB
        name: MeiweiPCB_training_exp_FP32_lr0.02_bs=2_larger_guided_mask_maskconf=0.85_2B_512x512
```

The training results will be saved under `${IMAGINAIRE_OUTPUT_ROOT}/<project>/<group>/<name>/`.
If `IMAGINAIRE_OUTPUT_ROOT` is not set, the default path will be `checkpoints/`.

If you are using a custom dataset, please ensure you also update the following settings:

1. Dataset Path
    - `dataloader_train.dataset.dataset_dir`
2. Anomaly Type
    - `dataloader_train.dataset.anomaly_types`
    - `model.config.ag_config.anomaly_embedding.anomaly_types`
    
    **Note**: The anomaly type format in YAML config files is a list of lists, where each inner list contains `[TEXTURE, ANOMALY_TYPE]`. Both sections must have the same anomaly types.
    
    Examples:
    - **MeiweiPCB** (single anomaly type):
      ```yaml
      dataloader_train:
        dataset:
          anomaly_types: 
            - 
              - PCB
              - defect
      model:
        config:
          ag_config:
            anomaly_embedding:
              anomaly_types:
                - 
                  - PCB
                  - defect
      ```
    
    - **Multiple Anomaly Types**:
      ```yaml
      dataloader_train:
        dataset:
          anomaly_types: 
            - 
              - TEXTURE_1
              - anomaly_type_1
            - 
              - TEXTURE_1
              - anomaly_type_2
            - 
              - TEXTURE_2
              - anomaly_type_1
      model:
        config:
          ag_config:
            anomaly_embedding:
              anomaly_types:
                - 
                  - TEXTURE_1
                  - anomaly_type_1
                - 
                  - TEXTURE_1
                  - anomaly_type_2
                - 
                  - TEXTURE_2
                  - anomaly_type_1
      ```

#### 1.1.2 Trainer Configuration

This notebook runs a quick example with the following configuration:

```yaml
    checkpoint:
        save_iter: 100
    trainer:
        max_iter: 200
        logging_iter: 10
        validation_iter: 50
        run_validation: True
    dataloader_val:
        dataset:
            input_data_path: ag_inference/validation.jsonl
```

- Training runs for 200 iterations.
- Checkpoints are saved every 100 iterations.
- Validate every 50 iterations
  - Images are generated from `<input_data_path>`. Please refer to ([Section 3.1](./3-generation.ipynb)) for detail configuration.
  - Results are saved to `${IMAGINAIRE_OUTPUT_ROOT}/<project>/<group>/<name>/valid/<iteration>/`
  - Validation includes generating images for visual inspection and computing FID, which measures the similarity between the feature distributions of generated and real images in the CRADIOv3-B feature space.

<font color="red">**Important**: This is a quick example, so the generated image quality may be suboptimal.
To improve quality, increase the number of training iterations.
Based on our validation on the MeiweiPCB dataset, we recommend 75,000 iterations for full training. (`trainer.max_iter: 75000`)</font>


In [ ]:
!cat {LOCAL_PROJECT_DIR}/ag_configs/MeiweiPCB_NVDINOV2_2B_512.yaml

### 1.2 Training the PAIDF AnomalyGen Modules

Ensure the following directories are available and correctly configured:
- {LOCAL_PROJECT_DIR}/data: Training dataset ([Section 0.2](./0-setup-cuda128.ipynb)).
- {LOCAL_PROJECT_DIR}/checkpoints: Required pretrained modules ([Section 0.3](./0-setup-cuda128.ipynb)).
- {LOCAL_PROJECT_DIR}/ag_configs: Training configuration files ([Section 1.1](./1-training.ipynb)).
- {IMAGINAIRE_OUTPUT_ROOT}/results: Directory where training results and logs will be saved.

**Note**: For full training, it is recommended to use the command line instead of the notebook, as notebook cells have limited logging capacity.

<font color="red">**Important**: This is a quick example, so the generated image quality may be suboptimal.
To improve quality, increase the number of training iterations.
Based on our validation on the MeiweiPCB dataset, we recommend 75,000 iterations for full training. (`trainer.max_iter: 75000`)</font>

<details>
<summary> <b> The equivalent command in the bash terminal. (Click to show) <b> </summary>

```bash
export IMAGINAIRE_OUTPUT_ROOT=./results && \
CUDA_HOME=$CONDA_PREFIX \
CUDA_VISIBLE_DEVICES=0 \
torchrun --nproc_per_node=1 --master_port=12341 -m scripts.anomaly_gen.ag_train \
--config=cosmos_predict2/configs/base/ag_config.py \
--ag_config=ag_configs/MeiweiPCB_NVDINOV2_2B_512.yaml \
-- experiment=predict2_anomaly_gen_ddp_2b
```

</details>

In [ ]:
# Since `--live-stream` doesn't work well with `scripts.anomaly_gen.ag_train`, you should avoid `--live-stream` and instead check the stdout log here:
# results/anomaly_gen/MeiweiPCB/MeiweiPCB_training_exp_FP32_lr0.02_bs=2_larger_guided_mask_maskconf=0.85_2B_512x512/stdout.log
!conda run -n cosmos-predict2 \
    bash -c "export IMAGINAIRE_OUTPUT_ROOT=./results && \
        CUDA_HOME=\$CONDA_PREFIX \
        CUDA_VISIBLE_DEVICES=0 \
        torchrun --nproc_per_node=1 --master_port=12341 -m scripts.anomaly_gen.ag_train \
        --config=cosmos_predict2/configs/base/ag_config.py \
        --ag_config=ag_configs/MeiweiPCB_NVDINOV2_2B_512.yaml \
        -- experiment=predict2_anomaly_gen_ddp_2b"

Validation outputs are stored under `${IMAGINAIRE_OUTPUT_ROOT}/<project>/<group>/<name>/valid/<step>/`.
  - Subfolders:
    - `original_image/`: Original input images.
    - `original_mask/`: Original input masks.
    - `cropped_image/`: Original crops around input masked region.
    - `cropped_mask/`: Cropped masks aligned with each crop.
    - `annotated_image/`: Original images overlaid with cropped regions.
    - `mask_cropped_image/`: Cropped images with background masked out (only the masked regions remain visible), used for KPI computation.
    - `reconstructed_image/`: Images inpainted with learned anomaly.
  - `valid_kpi.csv`: Table summarizing anomaly-wise KPIs for the validation step.

**Example outputs at step 200** (MeiweiPCB, PCB+defect):

<font color="red">**Important**: This is a quick example, so the generated (reconstructed) image quality may be suboptimal.
To improve quality, increase the number of training iterations.
Based on our validation on the MeiweiPCB dataset, we recommend 75,000 iterations for full training. (`trainer.max_iter: 75000`)</font>

<table>
<tr>
  <th align="center">Image Name</th>
  <th align="center">Original Image</th>
  <th align="center">Original Mask</th>
  <th align="center">Cropped Image</th>
  <th align="center">Cropped Mask</th>
  <th align="center">Annotated Image</th>
  <th align="center">Reconstructed Image</th>
</tr>
<tr>
  <td><sub>PCB+defect_00000</sub></td>
  <td><img src="../../assets/anomaly_gen/training_example_materials/original_image/PCB+defect_00000.png" width="160"/></td>
  <td><img src="../../assets/anomaly_gen/training_example_materials/original_mask/PCB+defect_00000.png" width="160"/></td>
  <td><img src="../../assets/anomaly_gen/training_example_materials/cropped_image/PCB+defect_00000_00000.png" width="160"/></td>
  <td><img src="../../assets/anomaly_gen/training_example_materials/cropped_mask/PCB+defect_00000_00000.png" width="160"/></td>
  <td><img src="../../assets/anomaly_gen/training_example_materials/annotated_image/PCB+defect_00000_00000.png" width="160"/></td>
  <td><img src="../../assets/anomaly_gen/training_example_materials/reconstructed_image/PCB+defect_00000.png" width="160"/></td>
</tr>
<tr>
  <td><sub>PCB+defect_00001</sub></td>
  <td><img src="../../assets/anomaly_gen/training_example_materials/original_image/PCB+defect_00001.png" width="160"/></td>
  <td><img src="../../assets/anomaly_gen/training_example_materials/original_mask/PCB+defect_00001.png" width="160"/></td>
  <td><img src="../../assets/anomaly_gen/training_example_materials/cropped_image/PCB+defect_00001_00000.png" width="160"/></td>
  <td><img src="../../assets/anomaly_gen/training_example_materials/cropped_mask/PCB+defect_00001_00000.png" width="160"/></td>
  <td><img src="../../assets/anomaly_gen/training_example_materials/annotated_image/PCB+defect_00001_00000.png" width="160"/></td>
  <td><img src="../../assets/anomaly_gen/training_example_materials/reconstructed_image/PCB+defect_00001.png" width="160"/></td>
</tr>
<tr>
  <td><sub>PCB+defect_00002</sub></td>
  <td><img src="../../assets/anomaly_gen/training_example_materials/original_image/PCB+defect_00002.png" width="160"/></td>
  <td><img src="../../assets/anomaly_gen/training_example_materials/original_mask/PCB+defect_00002.png" width="160"/></td>
  <td><img src="../../assets/anomaly_gen/training_example_materials/cropped_image/PCB+defect_00002_00000.png" width="160"/></td>
  <td><img src="../../assets/anomaly_gen/training_example_materials/cropped_mask/PCB+defect_00002_00000.png" width="160"/></td>
  <td><img src="../../assets/anomaly_gen/training_example_materials/annotated_image/PCB+defect_00002_00000.png" width="160"/></td>
  <td><img src="../../assets/anomaly_gen/training_example_materials/reconstructed_image/PCB+defect_00002.png" width="160"/></td>
</tr>
</table>

### 1.3 (Optional) Visualization of Validation Curve

We can visualize the training progress by plotting FID curves across validation steps during training. This helps monitor model performance and convergence.
Make sure the plotting configuration aligns with the settings in Section 1.1.2:
- `--root`: `${IMAGINAIRE_OUTPUT_ROOT}/<project>/<group>/<name>/valid/`
- `--validation_iter`: Should match `<validation_iter>` in the training configuration
- `--max_iter`: Should match `<max_iter>` in the training configuration
- `--output_dir`: Path to save the visualization results
- `--anomaly_types`: List of anomaly types to visualize. Must include all types used in training.
Format: `TEXTURE+ANOMALY_TYPE`, e.g., `PCB+defect`.

<font color="red">**Important**: FID measures the difference in mean and variance within the CRADIOv3-B feature space between generated and real images. A lower FID indicates closer distributional similarity but does not guarantee better downstream task performance. Treat it only as a reference metric, not an absolute indicator of model quality.

In [ ]:
!conda run -n cosmos-predict2 \
    bash -c "python -m scripts.anomaly_gen.visualize \
         --root results/anomaly_gen/MeiweiPCB/MeiweiPCB_training_exp_FP32_lr0.02_bs=2_larger_guided_mask_maskconf=0.85_2B_512x512/valid \
        --output_dir plots/ \
        --validation_iter 50 \
        --max_iter 200 \
        --anomaly_types PCB+defect"


## Next Step

You can now proceed to the next step to utilize the automatic mask placement feature: [2-optional-auto-mask-placement.ipynb](./2-optional-auto-mask-placement.ipynb).

Alternatively, you may skip this step and directly to the generation step: [3-generation.ipynb](./3-generation.ipynb).